In [ ]:
import sys 
os.getcwd() 

sys.path.append('/home/huuthanhvy.nguyen001/LLMP') 
import LLMP as L


In [3]:
import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoProcessor, BitsAndBytesConfig, LlavaNextForConditionalGeneration, AutoModelForVision2Seq

# Hugging Face model id
model_id = "meta-llama/Llama-3.2-11B-Vision" 

# BitsAndBytesConfig int-4 config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, 
    bnb_4bit_use_double_quant=True, 
    bnb_4bit_quant_type="nf4", 
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load model and tokenizer
model = AutoModelForVision2Seq.from_pretrained(
    model_id,
    #use_cache=False,
    device_map="auto",
    # attn_implementation="flash_attention_2", # not supported for training
    torch_dtype=torch.bfloat16,
    quantization_config=bnb_config,
    low_cpu_mem_usage=True 
)

# Define LoRA config based on QLoRA experiments
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=4,
    bias="none",
    target_modules=["q_proj", "v_proj"],  # LoRA targets these transformer layers
    task_type="CAUSAL_LM",  # Task type for causal language modeling
)

# Apply LoRA adapters to the loaded model
model = get_peft_model(model, peft_config)

# Load the processor
processor = AutoProcessor.from_pretrained(model_id)


/home/huuthanhvy.nguyen001/anaconda3/envs/pytorch/lib/python3.11/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/home/huuthanhvy.nguyen001/anaconda3/envs/pytorch/lib/python3.11/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
The model weights are not tied. Please use the `tie_weights` method before using the `infer_auto_device` function.


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [4]:
import os
import json

# Define the path to the train dataset
train_dataset_path = os.path.join('finetuningDataset', 'train', 'dataset.json')

# Load the dataset into a variable
with open(train_dataset_path, 'r') as file:
    train_data = json.load(file)

# Now 'train_data' holds your training dataset
print("Training data loaded successfully!")

import os
import json

# Define the path to the train dataset
validation_dataset_path = os.path.join('finetuningDataset', 'validation', 'dataset.json')

# Load the dataset into a variable
with open(validation_dataset_path, 'r') as file:
    validation_dataset_path = json.load(file)

# Now 'train_data' holds your training dataset
print("Validation data loaded successfully!")

train_data[0]


Training data loaded successfully!
Validation data loaded successfully!


{'id': 'd0d225a3-b101-47c6-8d26-a8e1f7632e69',
 'image': 'd0d225a3-b101-47c6-8d26-a8e1f7632e69.jpg',
 'conversations': [{'from': 'human',
   'value': 'What do you see? If you see a simple line drawing that forms an acute angle, please estimate the size of the angle. Please respond with a possible range not larger than 10 degrees and report just the numbers.'},
  {'from': 'llms', 'value': '49'}]}

In [8]:
# Example usage for training
image_directory = '/home/huuthanhvy.nguyen001/LLMP/LLMP/finetuningDataset/images'

def collate_fn(examples, processor, image_dir):
    """
    Custom collate function to process text and image pairs.
    
    Args:
    examples: List of examples where each example contains messages and an image path.
    processor: Processor instance to handle tokenization and image processing.
    image_dir: Directory to fetch images.
    
    Returns:
    A dictionary containing input tensors for model training.
    """
    texts = []
    image_inputs = []
    
    valid_examples = []
    for example in examples:
        
        # Prepare the conversation messages
        system_message = "You are a helpful assistant."
        question = example["conversations"][0]["value"] if "conversations" in example and len(example["conversations"]) > 0 else ""
        answer = example["conversations"][1]["value"] if "conversations" in example and len(example["conversations"]) > 1 else ""

        
        # Construct the text manually to ensure correct placement of tokens
        text = (
            f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
            f"{system_message}\n"
            f"<|start_header_id|>user<|end_header_id|>\n"
            f"{question}\n"
            f"<|image|>\n"
            f"<|start_header_id|>assistant<|end_header_id|>\n"
            f"{answer}\n"
            f"<|eot_id|>"
        )
        texts.append(text.strip())

    # Ensure the number of image tokens matches the number of provided images
    for idx, text in enumerate(texts):
        image_token_count = text.count("<|image|>")
        if image_token_count != 1:
            print(f"Debug: Text {idx} contains {image_token_count} <|image|> tokens.")
            print(f"Debug: Text content - {text}")
            raise ValueError(f"Each text should contain exactly one <|image|> token, but found {image_token_count}")
    
    # Tokenize the texts and process the images
    batch = processor(text=texts, images=image_inputs, return_tensors="pt", padding=True)

    # The labels are the input_ids, and we mask the padding tokens in the loss computation
    labels = batch["input_ids"].clone() if "input_ids" in batch else torch.tensor([])
    if labels.numel() > 0:
        labels[labels == processor.tokenizer.pad_token_id] = -100  # Mask padding tokens
        
        # Ignore the image token index in the loss computation
        image_tokens = [processor.tokenizer.convert_tokens_to_ids(processor.image_token)]
        for image_token_id in image_tokens:
            labels[labels == image_token_id] = -100
        batch["labels"] = labels

    return batch

data_collator = lambda examples: collate_fn(examples, processor, image_directory)

In [10]:
from transformers import TrainingArguments, Trainer


#Modify TrainingArguments to reduce memory usage
training_args = TrainingArguments(
    output_dir="rami",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    num_train_epochs=1,
    learning_rate=5e-05,
    fp16=True,  # Enable mixed precision to save memory
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_data,
    eval_dataset=validation_dataset_path,
)

trainer.train()

/home/huuthanhvy.nguyen001/anaconda3/envs/pytorch/lib/python3.11/site-packages/accelerate/accelerator.py:500: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


[2024-10-03 22:47:14,180] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/huuthanhvy.nguyen001/anaconda3/envs/pytorch/bin/../lib/gcc/x86_64-conda-linux-gnu/11.2.0/../../../../x86_64-conda-linux-gnu/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/home/huuthanhvy.nguyen001/anaconda3/envs/pytorch/bin/../lib/gcc/x86_64-conda-linux-gnu/11.2.0/../../../../x86_64-conda-linux-gnu/bin/ld: cannot find -lcufile: No such file or directory
collect2: error: ld returned 1 exit status
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend. Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: rami-nguyen12 (ramihuunguyen). Use `wandb login --relogin` to force relogin


ValueError: Invalid input type. Must be a single image, a list of images, or a list of batches of images.

In [ ]:
import os
print(os.listdir('/home/huuthanhvy.nguyen001/LLMP/LLMP/finetuningDataset/images/'))

In [ ]:
!nvidia-smi

In [ ]:
!kill -9 2850727
